In [37]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [38]:
# read in all of the words
words = open('../names.txt', 'r').read().splitlines()
print('first few:',words[:8])
print('longest word:', max(len(w) for w in words))
print('total words:', len(words))

first few: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
longest word: 15
total words: 32033


In [39]:
# build the vocabulary of characters and mappings to/from integers
import string
START_STOP_TOKEN = '.'
tokens = [START_STOP_TOKEN, *string.ascii_lowercase]
stoi = { s: i for i, s in enumerate(tokens) }
itos = { i: s for s, i in stoi.items() }
N_TOKENS = len(tokens)
print(itos)
print('count of tokens:', N_TOKENS)

{0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z'}
count of tokens: 27


In [40]:
# build the dataset

BLOCK_SIZE = 3 # context length; number of chars used to predict the next ones
SPLIT_1 = 0.8 # 80% train
SPLIT_2 = 0.9 # 10% dev, 10% test

def build_dataset(words):
    X, Y = [], []
    for w in words:
        #print(); print(w)
        context = [0] * BLOCK_SIZE
        for ch in w + START_STOP_TOKEN:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '-->', itos[ix])
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

# -- with train, dev, test split --
import random
random.seed(42)
shuffled_words = words[:]
random.shuffle(shuffled_words)

n1 = int(SPLIT_1 * len(shuffled_words))
n2 = int(SPLIT_2 * len(shuffled_words))

print('trainset:', end=' ')
Xtr, Ytr = build_dataset(shuffled_words[:n1])
print('devset  :', end=' ')
Xdev, Ydev = build_dataset(shuffled_words[n1:n2])
print('testset :', end=' ')
Xte, Yte = build_dataset(shuffled_words[n2:])

trainset: torch.Size([182625, 3]) torch.Size([182625])
devset  : torch.Size([22655, 3]) torch.Size([22655])
testset : torch.Size([22866, 3]) torch.Size([22866])


In [41]:
# -- All the blocks so far were unchanged from the previous lecture ---
# Now time for new stuff

In [42]:
# helper function for double-checking manual gradients against PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact {str(ex):5s} | approx: {str(app):5s} | maxdiff: {maxdiff}')

In [57]:
N_EMBED = 10 # dimensionality of the char embedding vectors
INPUT_SIZE = BLOCK_SIZE * N_EMBED
N_HIDDEN = 200 # number of neurons in hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)
randn = lambda *size: torch.randn(size, generator=g)
kaiming_scale = lambda fan_in: (5/3)/(fan_in**0.5)

# Note: We are doing some non-standard initializations, as sometimes initializing
# with e.g. all zeros could mask an incorrect backprop implementation.

C =  randn(N_TOKENS, N_EMBED)
# -- Layer 1 --
W1 = randn(INPUT_SIZE, N_HIDDEN) * kaiming_scale(INPUT_SIZE)
b1 = randn(N_HIDDEN)             * 0.1 # keep b1 so we can check its grad, even tho batch norm makes it useless
# -- Layer 2 --
W2 = randn(N_HIDDEN, N_TOKENS)   * 0.1
b2 = randn(N_TOKENS)             * 0.1

# BatchNorm params
bn_gain = randn(1, N_HIDDEN) * 0.1 + 1.0
bn_bias = randn(1, N_HIDDEN) * .01

bn_mean_running = torch.zeros((1, N_HIDDEN))
bn_std_running = torch.ones((1, N_HIDDEN))
 
parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
for p in parameters:
    p.requires_grad = True

print('num params:', sum(p.nelement() for p in parameters))

num params: 12297


In [58]:
BATCH_SIZE = 32
n = BATCH_SIZE # for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (BATCH_SIZE,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

In [59]:
# forward pass, "chunked" into smaller steps that are easier to backward one at a time

emb = C[Xb] # embed characters into vectors
embcat = emb.view(emb.shape[0], -1) # concat the vectors

# Linear Layer 1
h_pre_bn = embcat @ W1 + b1 # hidden layer pre-activation

# BatchNorm Layer
bn_meani = 1/n * h_pre_bn.sum(0, keepdim=True)
bn_diff = h_pre_bn - bn_meani
bn_diff2 = bn_diff**2
bn_var = 1/(n-1) * bn_diff2.sum(0, keepdim=True) # Note: Bessel's correction (dividing by n-1, not n)
bn_var_inv = (bn_var + 1e-5)**(-0.5)
bn_raw = bn_diff * bn_var_inv
h_pre_act = bn_gain * bn_raw + bn_bias

# Non-linearity
h = torch.tanh(h_pre_act) # hidden layer

# Linear Layer 2
logits = h @ W2 + b2

# Cross entropy loss (same as F.cross_entropy(logits, Yb))
logits_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logits_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum ** -1 # If we use (1.0 / counts_sum) instead then we can't get backprop to be exact...
probs = counts * counts_sum_inv
log_probs = probs.log()
loss = -log_probs[range(n), Yb].mean()

for p in parameters:
    p.grad = None
for t in [log_probs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logits_maxes, logits, h, h_pre_act, bn_raw,
          bn_var_inv, bn_var, bn_diff2, h_pre_bn, bn_meani, embcat, emb]:
    t.retain_grad()

loss.backward()
loss

tensor(3.9353, grad_fn=<NegBackward0>)

In [85]:
print(counts.shape)
print(counts.sum(1, keepdims=True).shape)

torch.Size([32, 27])
torch.Size([32, 1])


In [88]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)

d_log_probs = torch.zeros_like(log_probs)
d_log_probs[range(n), Yb] = -1.0 / n
d_probs = (1 / probs) * d_log_probs
d_counts_sum_inv = (counts * d_probs).sum(1, keepdim=True) # we sum because of the broadcasting
d_counts = counts_sum_inv * d_probs
d_counts_sum = (-1 * (counts_sum ** -2)) * d_counts_sum_inv
d_counts += torch.ones_like(counts) * d_counts_sum

# -----------------

cmp('logprobs', d_log_probs, log_probs)
cmp('probs', d_probs, probs)
cmp('counts_sum_inv', d_counts_sum_inv, counts_sum_inv)
cmp('counts_sum', d_counts_sum, counts_sum)
cmp('counts', d_counts, counts)

# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

logprobs        | exact True  | approx: True  | maxdiff: 0.0
probs           | exact True  | approx: True  | maxdiff: 0.0
counts_sum_inv  | exact True  | approx: True  | maxdiff: 0.0
counts_sum      | exact True  | approx: True  | maxdiff: 0.0
counts          | exact True  | approx: True  | maxdiff: 0.0
